# Módulo nulos
En este dataset un nulo casi nunca es un dato perdido: es **estructural** (la columna no
aplica a esa materia, ámbito, tipo de fila u órgano). Este módulo clasifica cada columna
por el motivo de su ausencia y cada fila por su estado de gestión.

In [ ]:
import numpy as np
import pandas as pd

CAT_COMPLETA = "completa"
CAT_ESTRUCTURAL_MATERIA = "estructural_materia"
CAT_ESTRUCTURAL_GEOGRAFICO = "estructural_geografico"
CAT_ESTRUCTURAL_RECURSOS = "estructural_recursos"
CAT_ESTRUCTURAL_ELEMENTO = "estructural_tipo_elemento"
CAT_PROCESAL_EN_TRAMITE = "procesal_en_tramite"

COLUMNAS_GEOGRAFICAS = {"ciudad", "distrito"}
COLUMNAS_RECURSOS = {"recurso_tribunales_publicados", "recurso_salas_publicadas", "recurso_otros_organos_publicados"}
COLUMNAS_CIVIL_EXCLUSIVAS = {"readecuadas_ley_439", "preliminares_formalizados", "cautelares_formalizados"}
COLUMNAS_PENAL_EXCLUSIVAS = {
    "concluidas_rechazo_denuncia", "merecieron_imputacion_formal", "procesos_rebeldia",
    "merecieron_acusacion", "concluidas_otras_formas", "imputacion_directa_procedimiento_inmediato",
    "otras_formas_finalizacion", "recibidas_declinatoria_inhibitoria", "ingresadas_conversion_acciones",
    "otras_formas_conclusion", "resueltas_sentencia", "ingresadas_reenvio", "otras_formas_ingreso",
    "remitidas_excusa_recusacion", "remitidas_otras_formas", "tipo_accion_penal",
}
COLUMNAS_FORMULARIOS_EXTERNOS = {
    "conciliacion", "num_juzgados", "recibidas_otros_juzgados", "reparacion_dano_conciliacion",
    "remision_art_299_ley_548", "remitidas_finalizacion_competencia", "reparacion_dano",
    "sobreseimiento", "terminacion_anticipada", "concluidas_extincion_prescripcion",
    "concluidas_sentencia_juicio",
}

In [ ]:
def categorizar_columna(columna, n_nulos, total_filas):
    if n_nulos == 0:
        return CAT_COMPLETA, "100% poblada; sin valores nulos"
    if columna in COLUMNAS_GEOGRAFICAS:
        return CAT_ESTRUCTURAL_GEOGRAFICO, "Excluyente por ámbito: 'ciudad' solo en capitales/El Alto, 'distrito' solo en provincias"
    if columna in COLUMNAS_RECURSOS:
        return CAT_ESTRUCTURAL_RECURSOS, "Blancos publicados en el Anuario donde no existen salas, tribunales u otros órganos creados en esa localidad"
    if columna in COLUMNAS_FORMULARIOS_EXTERNOS:
        return CAT_ESTRUCTURAL_MATERIA, "Columna correspondiente a layouts de resolución o de otros cuadros que no aplica a causas por tipo de proceso (0 registros)"
    if columna in COLUMNAS_CIVIL_EXCLUSIVAS:
        return CAT_ESTRUCTURAL_MATERIA, "Exclusiva de los cuadros civiles (5.1.1.1 / 6.1.1.1) bajo el Código Procesal Civil (Ley 439); nula por ley en penal/familiar"
    if columna in COLUMNAS_PENAL_EXCLUSIVAS:
        return CAT_ESTRUCTURAL_MATERIA, "Exclusiva de cuadros penales (instrucción o sentencia); nula por ley en materias civiles/sociales"
    if columna == "resueltas":
        return CAT_ESTRUCTURAL_ELEMENTO, "Nula exclusivamente en 285 filas de encabezados padre ('accion_penal' y 'otro_detalle'). 100% poblada en las 1.655 filas procesales"
    if columna == "pendientes_inicio":
        return CAT_ESTRUCTURAL_MATERIA, "Nula en los cuadros civiles 5.1.1.1 y 6.1.1.1 donde el Anuario publicó readecuadas Ley 439 en lugar de stock inicial"
    if columna == "grupo_proceso" or columna == "grupo_proceso_norm":
        return CAT_ESTRUCTURAL_MATERIA, "Etiquetas rotadas publicadas únicamente en cuadros civiles y familiares con agrupación formal de procesos"
    if columna == "remitidas_otros_juzgados":
        return CAT_ESTRUCTURAL_MATERIA, "Forma de salida penal publicada solo en juzgados de instrucción y sentencia penal"
    pct = n_nulos / total_filas
    return "otra_ausencia", "Ausencia observada en " + str(n_nulos) + " filas ({:.1%})".format(pct)


def clasificar_matriz_nulos(df):
    total_filas = len(df)
    registros = []
    for col in df.columns:
        n_nulos = int(df[col].isnull().sum())
        categoria, explicacion = categorizar_columna(col, n_nulos, total_filas)
        registros.append({
            "columna": col,
            "total_filas": total_filas,
            "n_nulos": n_nulos,
            "pct_nulos": float(n_nulos / total_filas),
            "n_unicos": int(df[col].nunique(dropna=True)),
            "tipo_dato": str(df[col].dtype),
            "categoria_nulo": categoria,
            "explicacion": explicacion,
        })
    return pd.DataFrame(registros).sort_values(by=["pct_nulos", "columna"], ascending=[False, True])


def clasificar_dinamica_procesal(df):
    # en_tramite_exclusivo: atendió pero no resolvió nada; con_resolucion_parcial; resolucion_total;
    # sin_movimiento: nada atendido; encabezado_o_detalle: filas que no son procesos.
    estados = pd.Series("indeterminado", index=df.index, dtype="string")
    es_proceso = df["tipo_elemento_analitico"] == "proceso"
    es_no_proceso = df["tipo_elemento_analitico"].isin(["accion_penal", "otro_detalle"])
    estados[es_no_proceso] = "encabezado_o_detalle"

    atendidas = df["atendidas"].fillna(0)
    resueltas = df["resueltas"].fillna(0)
    estados[es_proceso & (atendidas == 0)] = "sin_movimiento"
    estados[es_proceso & (atendidas > 0) & (resueltas == 0)] = "en_tramite_exclusivo"
    estados[es_proceso & (resueltas > 0) & (resueltas < atendidas)] = "con_resolucion_parcial"
    estados[es_proceso & (atendidas > 0) & (resueltas >= atendidas)] = "resolucion_total"
    return estados